# Causal Boundaries — Paper 1 Section 3 (Comprehensive Intervention)

**Goal**: map the full causal phase-space of single-direction probe patching in Qwen3.6-35B-A3B.

Per-rollout factorial design (8 generations × 20 rollouts = 160 gens, ~4h on RTX 6000 Blackwell 96GB):

1. **Baseline** — no patch (control)
2. **Alpha sweep @ T=10, L17**: α ∈ {20, 40} — does stronger push flip? (α=12 gave +5pp in v2)
3. **T sweep @ α=20, L17**: T ∈ {30, 100} — is mid/late patch easier than silent-plan T=10?
4. **Ablation** — zero out source direction projection (no target) — tests *necessity*
5. **Null** — random direction, matched norm to α=20 — controls for noise

**Verdicts produced**:
- Alpha→effect curve (does effect saturate, explode, or stay null?)
- T→effect curve (three-phase commitment — is mid-plan more patchable?)
- Ablation → accuracy drop (is direction *necessary* even if not sufficient to flip?)

Hook is installed at **L17** (best detection layer). Probe comes from Stage B L17 refit (85.7% train fit).

## Cell 1 — Setup (install, auth)

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/causal_boundaries')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
PATCH_LAYER = 17  # L17 is our best layer for both detection AND V2 intervention

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Load Stage B + fit L17 probe (single pass, 2-3 min)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

PROBE_CACHE = OUT / 'probe_L17.pkl'
if PROBE_CACHE.exists():
    with open(PROBE_CACHE, 'rb') as f:
        P = pickle.load(f)
    scaler, pca, logreg = P['scaler'], P['pca'], P['logreg']
    directions_by_letter = P['directions']
    rollout_info = P['rollouts']
    response_tokens_all = P['tokens']
    letters = P['letters']
    print(f'loaded cached L17 probe + {len(rollout_info)} rollouts')
else:
    corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                                repo_type='dataset', cache_dir='/content/cache')
    shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
    print(f'{len(shards)} shards')

    MIN_LEN = 200
    POOL_WINDOW = 10
    pooled = []; labels_letter = []; rollout_info = []; response_tokens_all = []
    for shard in tqdm(shards, desc='L17 probe-fit'):
        with safe_open(str(shard), framework='pt') as f:
            meta = f.metadata()
            offs = json.loads(meta['offsets'])[f'L{PATCH_LAYER}']
            rollouts = json.loads(meta['rollouts'])
            rts = json.loads(meta['response_tokens'])
            q_options = json.loads(meta['options'])
            q_idx = int(meta['question_global_idx'])
            acts = f.get_tensor(f'acts_L{PATCH_LAYER}')
            for r_idx, r in enumerate(rollouts):
                if r['pred'] is None or r['response_len'] < MIN_LEN:
                    continue
                rollout_acts = acts[offs[r_idx]:offs[r_idx+1]].float().numpy()
                pooled.append(rollout_acts[:POOL_WINDOW].mean(axis=0))
                labels_letter.append(r['pred'])
                rollout_info.append(dict(
                    question=meta['question'], options=q_options, gold=meta['gold_letter'],
                    pred=r['pred'], correct=r['correct'], question_idx=q_idx, rollout_id=r['rollout_id']))
                response_tokens_all.append(rts[r_idx])

    pooled = np.stack(pooled)
    labels_letter = np.array(labels_letter)
    letters = sorted(set(labels_letter))
    print(f'Extracted {len(pooled)} rollouts; letters: {letters}')

    letter_to_int = {l: i for i, l in enumerate(letters)}
    y = np.array([letter_to_int[l] for l in labels_letter])
    scaler = StandardScaler().fit(pooled)
    pca = PCA(n_components=128, random_state=42).fit(scaler.transform(pooled))
    logreg = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
        pca.transform(scaler.transform(pooled)), y)
    print(f'L17 probe fit score: {logreg.score(pca.transform(scaler.transform(pooled)), y):.3f}')

    directions_by_letter = {}
    for letter, int_idx in letter_to_int.items():
        coef_pca = logreg.coef_[int_idx]
        direction_scaled = pca.components_.T @ coef_pca
        direction_raw = direction_scaled * scaler.scale_
        direction_raw = direction_raw / (np.linalg.norm(direction_raw) + 1e-8)
        directions_by_letter[letter] = direction_raw.astype(np.float32)

    with open(PROBE_CACHE, 'wb') as f:
        pickle.dump(dict(scaler=scaler, pca=pca, logreg=logreg,
                         directions=directions_by_letter, labels=labels_letter,
                         rollouts=rollout_info, tokens=response_tokens_all,
                         letters=letters), f)
    print(f'cached probe + directions to {PROBE_CACHE}')

d_model = len(directions_by_letter[letters[0]])
print(f'd_model={d_model}, letters={letters}')

## Cell 4 — Hook + intervention function (ablation-aware)

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class PatchController:
    """Add patch_vector at absolute token position = prompt_len + target_response_pos.
    Supports ablation mode: project OUT the source direction instead of adding target-source.
    """
    def __init__(self):
        self.patch_vector = None
        self.ablate_direction = None  # unit vector to project out (ablation mode)
        self.prompt_len = None
        self.target_response_pos = None
        self.active = False
        self.applied = False

    def start_patch(self, patch_vec, prompt_len, target_response_pos):
        self.patch_vector = torch.from_numpy(patch_vec).to('cuda', torch.bfloat16)
        self.ablate_direction = None
        self.prompt_len = prompt_len
        self.target_response_pos = target_response_pos
        self.active = True
        self.applied = False

    def start_ablate(self, direction_unit, prompt_len, target_response_pos):
        self.ablate_direction = torch.from_numpy(direction_unit).to('cuda', torch.bfloat16)
        self.patch_vector = None
        self.prompt_len = prompt_len
        self.target_response_pos = target_response_pos
        self.active = True
        self.applied = False

    def stop(self):
        self.active = False
        self.patch_vector = None
        self.ablate_direction = None

    def make_hook(self):
        def hook(module, inp, out):
            if not self.active or self.applied: return out
            hidden = out[0] if isinstance(out, tuple) else out
            target_abs_pos = self.prompt_len + self.target_response_pos
            if hidden.shape[1] > target_abs_pos:
                hidden = hidden.clone()
                if self.patch_vector is not None:
                    hidden[:, target_abs_pos, :] += self.patch_vector
                elif self.ablate_direction is not None:
                    # project out: h - (h·d)d  where d is unit
                    d = self.ablate_direction
                    h_target = hidden[:, target_abs_pos, :]  # [B, d_model]
                    proj = (h_target * d).sum(dim=-1, keepdim=True) * d
                    hidden[:, target_abs_pos, :] = h_target - proj
                self.applied = True
                if isinstance(out, tuple):
                    return (hidden,) + out[1:]
                return hidden
            return out
        return hook

controller = PatchController()
layer_module = get_layer_module(model, PATCH_LAYER)
hook_handle = layer_module.register_forward_hook(controller.make_hook())
print(f'hook installed on L{PATCH_LAYER}')

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

def format_prompt(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason step by step, "
        "then put the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

def generate_with_intervention(prompt, forced_response_prefix_tokens, mode, data, max_new=1024):
    """mode ∈ {'baseline', 'patch', 'ablate'}
    data: patch_vec np.array (for 'patch') | direction_unit np.array (for 'ablate') | None (for 'baseline')
    """
    enc = tok(prompt, return_tensors='pt').to('cuda')
    prompt_len = enc.input_ids.shape[1]
    forced_ids = torch.tensor([forced_response_prefix_tokens], device='cuda')
    full_input = torch.cat([enc.input_ids, forced_ids], dim=1)
    full_attn = torch.ones_like(full_input)
    patch_pos = len(forced_response_prefix_tokens) - 1

    controller.stop()
    if mode == 'patch' and data is not None:
        controller.start_patch(data, prompt_len, patch_pos)
    elif mode == 'ablate' and data is not None:
        controller.start_ablate(data, prompt_len, patch_pos)
    try:
        with torch.no_grad():
            out = model.generate(
                input_ids=full_input, attention_mask=full_attn,
                max_new_tokens=max_new, do_sample=False,
                pad_token_id=tok.pad_token_id, use_cache=True,
            )
    finally:
        controller.stop()
    generated = out[0, full_input.shape[1]:]
    full_response = torch.cat([forced_ids[0], generated])
    return tok.decode(full_response, skip_special_tokens=True)

print('intervention function ready')

## Cell 5 — Factorial sweep (N=20 × 8 gens = 160 total, ~4h)

Per rollout:
1. **baseline** — no patch
2. **patch α=20 T=10** — moderate strength at silent plan
3. **patch α=40 T=10** — strong strength at silent plan
4. **patch α=20 T=30** — moderate strength at mid-plan (prose just starting)
5. **patch α=20 T=100** — moderate strength at late plan (textual commitment)
6. **ablate T=10** — zero out source direction at silent plan
7. **ablate T=100** — zero out source direction at late plan
8. **null** — random direction, norm matched to α=20

In [ ]:
correct_mask = np.array([r['correct'] for r in rollout_info], dtype=bool)
confident_idx = np.where(correct_mask)[0]
np.random.seed(42)
np.random.shuffle(confident_idx)

N_INTERVENTIONS = 20
CONFIGS = [
    ('baseline', None, None, None),
    ('patch_a20_t10', 'patch', 20.0, 10),
    ('patch_a40_t10', 'patch', 40.0, 10),
    ('patch_a20_t30', 'patch', 20.0, 30),
    ('patch_a20_t100', 'patch', 20.0, 100),
    ('ablate_t10', 'ablate', None, 10),
    ('ablate_t100', 'ablate', None, 100),
    ('null_a20_t10', 'null', 20.0, 10),
]

results = []
t0 = time.time()
for i, rollout_idx in enumerate(tqdm(confident_idx[:N_INTERVENTIONS], desc='causal-boundaries')):
    info = rollout_info[rollout_idx]
    source_letter = info['pred']
    letters_avail = [l for l in directions_by_letter if l != source_letter and ord(l)-ord('A') < len(info['options'])]
    if not letters_avail: continue
    target_letter = random.Random(int(rollout_idx)).choice(letters_avail)
    prompt = format_prompt(info['question'], info['options'])

    d_source = directions_by_letter[source_letter]
    d_target = directions_by_letter[target_letter]

    row = dict(
        idx=int(rollout_idx), source=source_letter, target=target_letter, gold=info['gold'],
        n_options=len(info['options']), question_idx=info['question_idx'],
    )

    for name, mode, alpha, t_pos in CONFIGS:
        if t_pos is not None and len(response_tokens_all[rollout_idx]) <= t_pos:
            row[name] = None
            continue
        forced_prefix = response_tokens_all[rollout_idx][:t_pos] if t_pos is not None else None

        if mode is None:
            # baseline: re-generate from T=10 with no patch
            forced_prefix = response_tokens_all[rollout_idx][:10]
            text = generate_with_intervention(prompt, forced_prefix, 'baseline', None)
        elif mode == 'patch':
            patch_vec = alpha * (d_target - d_source)
            text = generate_with_intervention(prompt, forced_prefix, 'patch', patch_vec)
        elif mode == 'ablate':
            text = generate_with_intervention(prompt, forced_prefix, 'ablate', d_source)
        elif mode == 'null':
            rng = np.random.RandomState(int(rollout_idx) * 31 + hash(name) % 1000)
            rand_dir = rng.randn(len(d_source)).astype(np.float32)
            rand_dir = rand_dir / np.linalg.norm(rand_dir) * alpha
            text = generate_with_intervention(prompt, forced_prefix, 'patch', rand_dir)

        row[name] = extract_letter(text, len(info['options']))

    results.append(row)
    if (i+1) % 4 == 0:
        # Quick progress summary
        tgt = {c[0]: sum(1 for r in results if r.get(c[0]) == r['target']) for c in CONFIGS if c[0] != 'baseline'}
        print(f'  [{i+1}/{N_INTERVENTIONS}] target hits: ' + ' | '.join(f'{k}={v}' for k,v in tgt.items()))

with open(OUT/'causal_boundaries.json', 'w') as f:
    json.dump(dict(layer=PATCH_LAYER, n=len(results),
                   configs=[c[0] for c in CONFIGS], results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 6 — Analysis: effect by config, curves, ablation

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

CONFIG_NAMES = [c[0] for c in CONFIGS]

def summarize(cname):
    c = Counter()
    for r in results:
        v = r.get(cname)
        if v is None: c['invalid'] += 1
        elif v == r['source']: c['source'] += 1
        elif v == r['target']: c['target'] += 1
        else: c['other'] += 1
    total = sum(c.values())
    return {k: (c.get(k,0), c.get(k,0)/total) for k in ['source','target','other','invalid']}

print('\n=== Effect by config ===')
print(f'{"config":22s} {"source%":>8s} {"target%":>8s} {"other%":>7s} {"inv%":>6s}')
for name in CONFIG_NAMES:
    s = summarize(name)
    print(f'{name:22s} {s["source"][1]*100:>7.1f}% {s["target"][1]*100:>7.1f}% {s["other"][1]*100:>6.1f}% {s["invalid"][1]*100:>5.1f}%')

# Key metric: target flip rate minus null flip rate
null_flip = summarize('null_a20_t10')['target'][1]
print(f'\nNull (random α=20) target flip rate: {null_flip*100:.1f}%')
print(f'\n=== Directional effect (flip_rate − null_flip_rate) ===')
for name in CONFIG_NAMES:
    if name in ('baseline','null_a20_t10'): continue
    s = summarize(name)
    effect = s['target'][1] - null_flip
    verdict = '⭐ strong' if effect > 0.30 else ('moderate' if effect > 0.15 else ('weak' if effect > 0.05 else 'null'))
    print(f'  {name:22s}  flip={s["target"][1]*100:>5.1f}%  effect={effect*100:+.1f}pp  [{verdict}]')

print(f'\n=== Ablation (source-direction projection) ===')
baseline_acc = summarize('baseline')['source'][1]
for name in ['ablate_t10','ablate_t100']:
    s = summarize(name)
    drop = baseline_acc - s['source'][1]
    print(f'  {name}: source kept {s["source"][1]*100:.1f}% (baseline {baseline_acc*100:.1f}% → drop {drop*100:+.1f}pp)')

# Plot curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Alpha curve at T=10
alphas = [0, 20, 40]
alpha_flip = [summarize('baseline')['target'][1]*100, summarize('patch_a20_t10')['target'][1]*100, summarize('patch_a40_t10')['target'][1]*100]
alpha_kept = [summarize('baseline')['source'][1]*100, summarize('patch_a20_t10')['source'][1]*100, summarize('patch_a40_t10')['source'][1]*100]
axes[0].plot(alphas, alpha_flip, 'o-', label='target flip', color='green')
axes[0].plot(alphas, alpha_kept, 'o-', label='source kept', color='blue')
axes[0].axhline(null_flip*100, linestyle='--', color='red', label=f'null flip ({null_flip*100:.0f}%)')
axes[0].set_xlabel('α (patch strength)'); axes[0].set_ylabel('% of rollouts')
axes[0].set_title('Alpha sweep @ T=10 L17'); axes[0].legend(); axes[0].grid(alpha=0.3)

# T curve at α=20
ts = [10, 30, 100]
t_flip = [summarize('patch_a20_t10')['target'][1]*100, summarize('patch_a20_t30')['target'][1]*100, summarize('patch_a20_t100')['target'][1]*100]
t_kept = [summarize('patch_a20_t10')['source'][1]*100, summarize('patch_a20_t30')['source'][1]*100, summarize('patch_a20_t100')['source'][1]*100]
axes[1].plot(ts, t_flip, 'o-', label='target flip', color='green')
axes[1].plot(ts, t_kept, 'o-', label='source kept', color='blue')
axes[1].set_xlabel('T (patch position in response)'); axes[1].set_ylabel('% of rollouts')
axes[1].set_title('T sweep @ α=20 L17'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Ablation
abl_names = ['baseline', 'ablate_t10', 'ablate_t100']
abl_source = [summarize(n)['source'][1]*100 for n in abl_names]
axes[2].bar(abl_names, abl_source, color=['gray','orange','red'])
axes[2].set_ylabel('source kept %')
axes[2].set_title('Ablation: is direction necessary?'); axes[2].grid(alpha=0.3, axis='y')
for i,v in enumerate(abl_source): axes[2].text(i, v+1, f'{v:.0f}%', ha='center')

plt.tight_layout()
plt.savefig(OUT/'causal_boundaries_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ saved plot to {OUT/"causal_boundaries_plots.png"}')